In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# Leemos Desde Los Archivos CSV

In [0]:
df_pato=(
    spark.read.option("header","true")
    .option("inferSchema","true")
    .option("delimiter",",")
    .csv("/Volumes/saps_ctes/bronze/source_system/consultas_por_patologia.csv")
)
display(df_pato.columns)
#df_pato.display(5)

In [0]:
df_pato_nuevo=df_pato.withColumnRenamed("Barrio del Operativo","barrio_operativo")
df_pato_nuevo.columns

# Escribimos los datos a la capa de bronce

In [0]:
df_pato_nuevo.write.mode("overwrite").saveAsTable("saps_ctes.bronze.datosctes_consultas_patologia")

# Controlo la carga de datos

In [0]:
%sql

SELECT * FROM saps_ctes.bronze.datosctes_consultas_patologia LIMIT 1

# Repito el proceso para cargar el resto de archivos

In [0]:
df_saps=(
    spark.read.option("header","true")
    .option("inferSchema","true")
    .option("delimiter",",")
    .csv("/Volumes/saps_ctes/bronze/source_system/listado_saps.csv")
)

df_inmu=(
    spark.read.option("header","true")
    .option("inferSchema","true")
    .option("delimiter",",")
    .csv("/Volumes/saps_ctes/bronze/source_system/inmunizaciones.csv")
)

# Corroboro que los datos se cargaron correctamente

In [0]:
display(df_inmu.limit(5))

In [0]:
display(df_saps.limit(5))

# Controlo los nombres de las columnas para que no haya problemas

In [0]:
df_saps.columns

In [0]:
df_inmu.columns

# Renombro las columnas quitando los espacios en blanco por "_"

In [0]:
from pyspark.sql.functions import col

# Renombrar dinámicamente todas las columnas
df_inmu_renamed = df_inmu.select([
    col(field.name).alias(field.name.replace(" ", "_")) 
    for field in df_inmu.schema.fields
])  
df_inmu_renamed.columns

# Cargo los datos a las tablas en la capa de bronce

In [0]:
#df_saps.write.mode("Overwrite").saveAsTable("saps_ctes.bronze.datosctes_saps")

#df_inmu_renamed.write.mode("Overwrite").saveAsTable("saps_ctes.bronze.datosctes_inmunizaciones")

# Reviso que los datos se cargaron en las tablas

In [0]:
%sql
SELECT
(SELECT COUNT(*) FROM saps_ctes.bronze.datosctes_inmunizaciones) as Total_Inmunizaciones,
(SELECT COUNT(*) FROM saps_ctes.bronze.datosctes_saps) as Total_Saps

In [0]:
from bronze_config import INGESTION_CONFIG
display(INGESTION_CONFIG)

In [0]:
%sql

USE CATALOG saps_ctes;
USE SCHEMA bronze;

In [0]:
#Esta importación es importante para trabajar con el código que me
#permite renombrar las columnas
from pyspark.sql.functions import col
for item in INGESTION_CONFIG:
    df=(
    spark.read.option("header","true")
    .option("inferSchema","true")
    .option("delimiter",item['delimiter'])
    .csv(item["path"])
    )

    df_renamed = df.select([
        col(field.name).alias(field.name.replace(" ", "_")) 
        for field in df.schema.fields
    ]) 

    df_renamed.write.mode("overwrite").saveAsTable(f"bronze.{item["table"]}")


In [0]:
%sql
DROP TABLE IF EXISTS saps_ctes.bronze.datosctes_inmunizaciones;
DROP TABLE IF EXISTS saps_ctes.bronze.datosctes_saps;
DROP TABLE IF EXISTS saps_ctes.bronze.datosctes_consultas_patologia